# Emotion Detection - YOLO11s Training on Kaggle (Dual GPU Mode)

### 🚀 Setup Instructions:
1. **Add Data**: Right sidebar ➔ **+ Add Data** ➔ Search `8-facial-expressions-for-yolo` ➔ Add.
2. **Accelerator**: Right sidebar ➔ **Accelerator** ➔ **GPU T4 x2**.
3. **Run All**: Click the 'Run All' button.

In [ ]:
!pip install -U ultralytics

In [ ]:
import os
import yaml

print("🔍 Searching for dataset...")

def find_data_yaml(root_path):
    if not os.path.exists(root_path):
        return None, None
    for root, dirs, files in os.walk(root_path):
        if 'data.yaml' in files:
            return os.path.join(root, 'data.yaml'), root
    return None, None

# Search paths
search_paths = [
    '/kaggle/input/datasets/aklimarimi/8-facial-expressions-for-yolo',
    '/kaggle/input/8-facial-expressions-for-yolo',
    '/kaggle/input'
]

found_yaml = None
for p in search_paths:
    found_yaml, data_path = find_data_yaml(p)
    if found_yaml: break

if found_yaml:
    print(f"✅ Found dataset at: {data_path}")
    with open(found_yaml, 'r') as f:
        config = yaml.safe_load(f)
    
    config['path'] = data_path
    config['train'] = 'train/images'
    config['val'] = 'valid/images'
    config['test'] = 'test/images'
    
    with open('/kaggle/working/emotion_data.yaml', 'w') as f:
        yaml.dump(config, f)
    print("✅ emotion_data.yaml generated.")
else:
    print("❌ ERROR: Dataset NOT found!")

In [ ]:
from ultralytics import YOLO
import torch

# Load YOLO11-Small (yolo11s.pt)
model = YOLO('yolo11s.pt')

# Auto-detect Dual GPU
gpu_count = torch.cuda.device_count()
devices = [i for i in range(gpu_count)] if gpu_count > 1 else 0
print(f"Using devices: {devices} ({'Dual GPU Mode! 🚀🚀' if gpu_count > 1 else 'Single GPU'})")

# Train the model
results = model.train(
    data='/kaggle/working/emotion_data.yaml',
    epochs=50,
    imgsz=640,
    batch=128 if gpu_count > 1 else 64, 
    device=devices,
    name='emotion_yolo11s_kaggle'
)